In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
import numpy as np
from pathlib import Path
from scipy.interpolate import make_smoothing_spline

from joblib import Parallel, delayed


# Load data

In [2]:
repo_folder = Path('../../..')
data_folder  = repo_folder / 'data'/ 'this_project' / '6_transporterKO'
metabolomics_folder = data_folder / 'D_big_screen'


In [3]:
normalization = 'TIC'
median_fn = metabolomics_folder / f'G_median_z_scores_{normalization}_norm_keio.csv'
df_data = pd.read_csv(median_fn)
df_data.rename(columns={'Z-score median': 'Median Z-score'}, inplace=True)

ionMz_annotation_fn = metabolomics_folder / 'H_ionMz_annotation.csv'
df_ionMz = pd.read_csv(ionMz_annotation_fn, index_col=0)

sample_metadata_fn = metabolomics_folder / 'I_sample_metadata_keio.csv'
df_sample_metadata = pd.read_csv(sample_metadata_fn, index_col=0)

In [4]:
df_w_mz = df_data.merge(df_ionMz, on='ionMz', how='left')

In [5]:
df = df_w_mz.merge(df_sample_metadata, on=['Batch-Tube', 'Timepoint'], how='left')

In [6]:
df.sort_values(by=['Batch-Tube', 'ionMz', 'Hours'], inplace=True)

# Compare the timeseries of each strain with WT

## Define functions

In [16]:
def spline_distance_and_std2(data, g1, g2, group = 'Strain', xval = 'AUC OD', 
                            yval = 'Median Z-score', 
                            ystd = 'Z-score std',
                            lam = 10, n_x_grid = 100, uniform_grid = False,
                            permutations = False):
    """
    Calculate the distance between two splines fitted to the data for two groups.
    Args:
        data (pd.DataFrame): DataFrame containing the data.
        g1 (str): First group to compare.
        g2 (str): Second group to compare (WT).
        group (str): Column name for grouping.
        xval (str): Column name for x-axis values.
        yval (str): Column name for y-axis values.
        ystd (str): Column name for standard deviation of y-axis values.
        lam (float): Smoothing parameter for spline fitting (10-100 works well).
        n_x_grid (int): Number of points in the x grid for interpolation.
    Returns:
        rel_distance (float): Relative distance between the two splines.
        mean_diff (float): Mean difference between the two splines.
        mae1 (float): Mean absolute error for the first group.
        mae2 (float): Mean absolute error for the second group.
        mean_mae (float): Mean absolute error averaged over both groups.
    """

    dg1 = data.loc[data[group] == g1]
    dg2 = data.loc[data[group] == g2]
    
    x1 = dg1[xval].values
    y1 = dg1[yval].values
    y1std = dg1[ystd].values 

    x2 = dg2[xval].values
    y2 = dg2[yval].values
    y2std = dg2[ystd].values

    if uniform_grid:
        # Define common x grid (e.g., union or intersection)
        x_common = np.linspace(max(x1.min(), x2.min()), min(x1.max(), x2.max()), n_x_grid, endpoint=True)
    else:
        x_min = max(x1.min(), x2.min())
        x_max = min(x1.max(), x2.max())
        x_common = np.sort([x for x in np.concatenate((x1, x2)) if x_min <= x <= x_max])

    # Spline interpolation
    w1 = 1/y1std
    w2 = 1/y2std

    spline1  =  make_smoothing_spline(x1, y1, w = w1, lam =lam)
    spline2  =  make_smoothing_spline(x2, y2, w = w2, lam =lam)


    # Estimate distance
    spline_distance = spline1(x_common)-spline2(x_common)
    # Absolute mean distance
    mean_abs_diff = np.mean(np.abs(spline_distance))
    # Mean distance
    mean_diff = np.mean(spline_distance)

    # Mean absolute error
    mae1 = np.mean(np.abs(spline1(x1) - y1))
    mae2 = np.mean(np.abs(spline2(x2) - y2))
    mean_mae = (mae1 + mae2) / 2.0

    # Relative distance
    rel_abs_distance = mean_abs_diff / mean_mae
    rel_distance = mean_diff / mean_mae
    

    if permutations:
        return rel_abs_distance, rel_distance
    else:
        return rel_abs_distance, rel_distance, mean_abs_diff, mean_diff, mae1, mae2, mean_mae



In [ ]:
def perform_permutations2(n_perms, data, g1, g2, 
                        xval, yval, lam = 10, bar = False):
    perm_stats_rel = []
    perm_stats_abs = []
    # Make sure there are no duplicated x-values at all
    is_duplicated = data[xval].duplicated()
    if np.any(is_duplicated):
        # Add tiny random noise to the x-values to break ties
        data.loc[is_duplicated, xval] += np.sort(np.random.normal(0, 1e-4, size=is_duplicated.sum()))

    data.sort_values(by = xval, inplace = True, ascending= True)


    loop = tqdm(range(n_perms)) if bar else range(n_perms)
    
    for _ in loop:
        data['Group'] = np.random.permutation(data['Strain'])
        rel_abs_dis, rel_dis = spline_distance_and_std2(data, g1 = g1, g2 = g2, group = 'Group', xval = xval,
                                        yval = yval, lam = lam, permutations = True)
        perm_stats_abs.append(rel_abs_dis)
        perm_stats_rel.append(rel_dis)
    
    return perm_stats_abs, perm_stats_rel
    
def process_strain_metabolite_2(strain, mz, dfmz, n_perms, xval = 'AUC OD', yval = 'Median Z-score',
                                lam = 10):
    data = dfmz.copy()
    # print(data['Strain'].unique())
    g1, g2 = sorted(data['Strain'].unique(), reverse=True)

    # Make sure there are no duplicated x-values for the same strain 
    is_duplicated = data[['Strain', xval]].duplicated()
    if np.any(is_duplicated):
        # Add tiny random noise to the x-values to break ties
        data.loc[is_duplicated, xval] += np.random.normal(0, 1e-4, size=is_duplicated.sum())
    data.sort_values(by = xval, inplace=True)

    rel_abs_distance, rel_distance, mean_abs_diff, mean_diff, mae1, mae2, mean_mae = spline_distance_and_std2(data,g1=g1, g2=g2, group = 'Strain', xval= xval,
                                                            yval=yval, lam = lam, permutations = False)
    
    if n_perms > 0:
        perm_stats_abs, perm_stats_rel = perform_permutations2(n_perms, data, g1, g2, 
                                          xval, yval, lam=lam)
        if rel_distance < 0:
            p_value_rel = np.sum(np.array(perm_stats_rel) <= rel_distance) / n_perms
        else:
            p_value_rel = np.sum(np.array(perm_stats_rel) >= rel_distance) / n_perms

        p_value_abs = np.sum(np.array(perm_stats_abs) >= rel_abs_distance) / n_perms
    else:
        p_value_rel = np.nan
        p_value_abs = np.nan
    return (strain, mz, rel_abs_distance, rel_distance, mean_abs_diff, mean_diff, mae1, mae2, mean_mae, g1, g2, p_value_rel, p_value_abs)

# Run screen

In [9]:
# Settings
xval = 'AUC OD'
yval = 'Median Z-score'
n_perms = 1000
n_jobs = 8
lam = 10
wt_strain = 'WT'


In [10]:
dis_data = {}
run = True


In [11]:
if run:
    all_ions = df.ionMz.unique()
    all_strains = df.Strain.unique()
    jobs = []
    for i in range(len(all_strains)):
        strain = all_strains[i]
        if strain == 'WT':
            continue
        batch = df.loc[df.Strain==strain, 'Batch'].values[0]
        idxs = (df.Batch == batch)&(df.Strain.isin([strain, wt_strain]))
        dfs = df.loc[idxs]
        for mz in all_ions:
            if dis_data.get((strain, mz)):
                continue
            dfmz = dfs.loc[dfs.ionMz==mz]
            if len(dfmz) > 10:
                jobs.append((strain, mz, dfmz))

    results = Parallel(n_jobs=n_jobs, verbose=1)(
        delayed(process_strain_metabolite_2)(strain, mz, dfi, n_perms = n_perms, lam = lam, xval=xval, yval=yval)
        for strain, mz, dfi in jobs
    )

    # dis_data = {}
    for res in results:
        (strain, mz, rel_abs_distance, rel_distance, mean_abs_diff, mean_diff, mae1, mae2, mean_mae, g1, g2, p_value_rel, p_value_abs) = res
        dis_data[(strain, mz)] = [rel_abs_distance, rel_distance, mean_abs_diff, mean_diff, mae1, mae2, mean_mae, g1, g2, p_value_rel, p_value_abs]

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:   15.3s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:  1.0min
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:  2.2min
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:  3.7min
[Parallel(n_jobs=8)]: Done 1234 tasks      | elapsed:  6.0min
[Parallel(n_jobs=8)]: Done 1784 tasks      | elapsed:  8.3min
[Parallel(n_jobs=8)]: Done 2434 tasks      | elapsed: 11.1min
[Parallel(n_jobs=8)]: Done 3184 tasks      | elapsed: 14.3min
[Parallel(n_jobs=8)]: Done 4034 tasks      | elapsed: 17.9min
[Parallel(n_jobs=8)]: Done 4984 tasks      | elapsed: 21.9min
/var/folders/xf/kl76knj11y72v0_qy4vv7tgh0000gp/T/ipykernel_79401/93999039.py:46: RuntimeWarning: divide by zero encountered in divide
/var/folders/xf/kl76knj11y72v0_qy4vv7tgh0000gp/T/ipykernel_79401/93999039.py:47: RuntimeWarning: divide by zero encountered in divide
/var/folders/xf/kl76knj11y72v0_qy4vv7tgh0

## Save data

In [12]:
if run:
    dis_df = pd.DataFrame(dis_data).T.reset_index()
    dis_df.columns = ['Strain', 'ionMz', 'Rel. abs. distance', 'Rel. distance', 'Mean abs. distance', 'Mean distance',
                    'MAE1', 'MAE2', 'Mean MAE', 'Strain 1', 'Strain 2', 'p-value rel', 'p-value abs']

In [13]:
dis_df.loc[dis_df['Strain']=='tolC', 'ionMz'].nunique()

281

In [14]:
distance_fn = metabolomics_folder / f'K_timecurve_comparison_{normalization}_norm_lam_{lam}_{xval}.csv'
dis_df.to_csv(distance_fn)

# Load data

In [15]:
dis_df = pd.read_csv(distance_fn, index_col=0)